In [1]:
import json
import logging
import urllib.request
from pathlib import Path
from typing import Any, Dict, List

In [2]:
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [3]:
# 전역 설정 (기존 인덱서 규격과 일치시킴)
COLLECTION_NAME = "robot_news_docs"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text:latest"
OLLAMA_GENERATE_MODEL = "llama3:latest"  # 사용 중이신 로컬 LLM 모델명으로 변경 가능 (예: gemma2, qwen2 등)
OLLAMA_API_URL = "http://localhost:11434/api"

In [4]:
def get_project_root() -> Path:
    """현재 파일 위치를 기준으로 프로젝트 루트 폴더를 탐색합니다."""
    current_file = Path(__file__).resolve()
    for parent in [current_file.parent, *current_file.parents]:
        if (parent / "vector_db").exists() or (parent / "outputs").exists():
            return parent
    return current_file.parents[1]

In [5]:
class SimpleOllamaClient:
    """Ollama API와의 통신을 담당하는 가벼운 클라이언트"""
    def __init__(self, base_url: str):
        self.base_url = base_url

    def get_embedding(self, text: str) -> List[float]:
        """질문을 벡터로 변환"""
        url = f"{self.base_url}/embeddings"
        body = json.dumps({"model": OLLAMA_EMBEDDING_MODEL, "prompt": text}, ensure_ascii=False).encode("utf-8")
        req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return [float(v) for v in data.get("embedding", [])]

    def generate_answer(self, prompt: str) -> str:
        """프롬프트를 받아 LLM 답변 생성 (Streaming 없이 일괄 반환)"""
        url = f"{self.base_url}/generate"
        body = json.dumps({
            "model": OLLAMA_GENERATE_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.2}  # 일관성 있고 사실에 기반한 답변을 위해 낮은 온도 설정
        }, ensure_ascii=False).encode("utf-8")
        
        req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data.get("response", "").strip()

In [6]:
def run_rag_pipeline(query_text: str):
    """Retrieval-Augmented Generation 전체 공정 수행"""
    try:
        import chromadb
    except ImportError:
        logger.error("chromadb가 설치되어 있지 않습니다. pip install chromadb 필요")
        return

    project_root = get_project_root()
    chroma_dir = project_root / "vector_db" / "robot_news_db"
    
    if not chroma_dir.exists():
        logger.error(f"DB를 찾을 수 없습니다: {chroma_dir}\n먼저 뉴스 빌더를 실행해 인덱스를 생성해 주세요.")
        return

    # 1. 인프라 및 클라이언트 초기화
    client = chromadb.PersistentClient(path=str(chroma_dir))
    ollama = SimpleOllamaClient(base_url=OLLAMA_API_URL)
    
    try:
        collection = client.get_collection(name=COLLECTION_NAME)
    except Exception:
        logger.error(f"Chroma 컬렉션 '{COLLECTION_NAME}'을 찾을 수 없습니다.")
        return

    print(f"\n🙋 사용자 질문: {query_text}")
    print("🔍 [STAGE 1] 질문 임베딩 생성 및 Chroma DB 기사 검색 중...")
    
    # 2. Retrieval: 질문을 벡터로 바꾸어 가장 관련성이 높은 기사 3개 추출
    try:
        query_vector = ollama.get_embedding(query_text)
    except Exception as e:
        logger.error(f"Ollama 임베딩 생성 실패. 서버 확인 필요: {e}")
        return

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=3,  # 연관 뉴스 상위 3개 추출
    )

    # 검색 결과 정제
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    if not documents or not documents[0]:
        print("❌ 관련된 뉴스 기사를 찾지 못했습니다. 일반 답변으로 전환하거나 다른 키워드로 검색하세요.")
        return

    print(f"📰 [STAGE 2] 관련 뉴스 {len(documents)}건 확보 완료.")
    
    # 문맥(Context) 및 출처 정보 조립
    context_str = ""
    sources = []
    for i, (doc, meta) in enumerate(zip(documents, metadatas), 1):
        context_str += f"[참조 기사 {i}: {meta.get('section_title')}]\n{doc}\n\n"
        sources.append(f"- {meta.get('section_title')} (출처 링크: {meta.get('action')})")

    # 3. Generation: 프롬프트 엔지니어링 (시스템 지침 + 뉴스 문맥 + 질문)
    system_prompt = (
        "당신은 로봇 산업 분석가입니다. 아래 제공된 [로봇신문 뉴스 기사 문맥]을 철저히 바탕으로 답변해 주세요.\n"
        "기사에 없는 가짜 정보(Hallucination)를 지어내지 말고 사실만을 기술해야 합니다. 한국어로 친절하게 답변하세요.\n\n"
        f"[로봇신문 뉴스 기사 문맥]\n{context_str}\n"
        f"[사용자 질문]\n{query_text}\n\n"
        "답변:"
    )

    print(f"🤖 [STAGE 3] {OLLAMA_GENERATE_MODEL} 모델을 통한 뉴스 기반 답변 생성 중...")
    try:
        answer = ollama.generate_answer(system_prompt)
        
        # 최종 결과 출력
        print("\n" + "="*60)
        print("✨ RAG 시스템 생성 답변:")
        print("="*60)
        print(answer)
        print("\n📌 답변 근거 뉴스 출처:")
        for source in sources:
            print(source)
        print("="*60 + "\n")
        
    except Exception as e:
        logger.error(f"LLM 답변 생성 중 오류 발생: {e}")

In [7]:
test_query = "삼성 관련 로봇 최신 뉴스 동향을 요약해 주고 어떤 기술 스택을 쓰는지 알려줘"
    
run_rag_pipeline(test_query)

NameError: name '__file__' is not defined